# ЛР3. Прокси-сервер

Тетрадь подготовлена на основе предыдущей ЛР и требований ЛР3.

In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import xmlrpc.client
import datetime
import threading
import time
from collections import deque

MAIN_SERVER_URL="http://localhost:8008"
STATS_SERVER_URL="http://localhost:8009"
RATE_LIMIT_N=5
RATE_LIMIT_T=5.0
main_server=xmlrpc.client.ServerProxy(MAIN_SERVER_URL,allow_none=True)
_stats=None
_times=deque()
_lock=threading.Lock()

class RequestHandler(SimpleXMLRPCRequestHandler): rpc_paths=('/RPC2',)

def _log(event_type,duration=None):
    global _stats
    try:
        if _stats is None: _stats=xmlrpc.client.ServerProxy(STATS_SERVER_URL,allow_none=True)
        _stats.log_event(event_type,datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f"),duration)
    except Exception as e:
        print(f"[proxy] stats unavailable: {e}")

def _acquire():
    while True:
        now=time.monotonic()
        with _lock:
            while _times and now-_times[0]>=RATE_LIMIT_T: _times.popleft()
            if len(_times)<RATE_LIMIT_N:
                _times.append(now); return
            wait=RATE_LIMIT_T-(now-_times[0])
        _log("too_many_requests")
        time.sleep(max(wait,0.01))

def _call(event_type,method,*args):
    _acquire(); start=time.perf_counter()
    try: return getattr(main_server,method)(*args)
    finally: _log(event_type,time.perf_counter()-start)

def ping(): return _call("ping","ping")
def now(): return _call("now","now")
def show_type(arg): return _call("show_type","show_type",arg)
def test_sum(a,b): return _call("test_sum","test_sum",a,b)
def test_pow(a,b): return _call("test_pow","test_pow",a,b)
def black_list_check(surname,name,patr,birth_date): return _call("black_list_check","black_list_check",surname,name,patr,birth_date)
def color_inversion(bin_data): return _call("color_inversion","color_inversion",bin_data)
def binarize_image(bin_data,threshold): return _call("binarize_image","binarize_image",bin_data,threshold)
def flip_vertical(bin_data): return _call("flip_vertical","flip_vertical",bin_data)
def cloudiness_percent(bin_data,threshold): return _call("cloudiness_percent","cloudiness_percent",bin_data,threshold)

server=SimpleXMLRPCServer(("localhost",8010),requestHandler=RequestHandler,allow_none=True)
server.register_introspection_functions()
server.register_function(ping,'ping')
server.register_function(now,'now')
server.register_function(show_type,'show_type')
server.register_function(test_sum,'test_sum')
server.register_function(test_pow,'test_pow')
server.register_function(black_list_check,'black_list_check')
server.register_function(color_inversion,'color_inversion')
server.register_function(binarize_image,'binarize_image')
server.register_function(flip_vertical,'flip_vertical')
server.register_function(cloudiness_percent,'cloudiness_percent')
print(f"Прокси: порт 8010; Rate limit={RATE_LIMIT_N}/{RATE_LIMIT_T:g} сек.")
server.serve_forever()


Прокси: порт 8010; Rate limit=5/5 сек.


127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:14:56] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:01] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:01] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:01] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:01] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:01] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:06] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:07] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:07] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 17:15:08] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202